# **SC : Practical - 9**

Mayuresh Vengurlekar

CMPN - B

23102B0018

In [1]:
!pip install pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('/content/stock.csv')

df['Return'] = df['Close'].pct_change()
df['MA'] = df['Close'].rolling(window=5).mean()
df = df.dropna()

class_names = {
    0: "DOWN (Sell Signal)",
    1: "UP (Buy Signal)"
}

df['Label'] = np.where(df['Return'] > 0, 1, 0)

features = ['Close', 'Return', 'MA']
X = df[features].values
y = df['Label'].values

print("Selected Features:", features)
print("Class Names:", class_names)

Selected Features: ['Close', 'Return', 'MA']
Class Names: {0: 'DOWN (Sell Signal)', 1: 'UP (Buy Signal)'}


In [4]:


scaler = MinMaxScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

class LVQ:
    def __init__(self, n_prototypes_per_class=1, learning_rate=0.01, epochs=50):
        self.n_prototypes_per_class = n_prototypes_per_class
        self.learning_rate = learning_rate
        self.epochs = epochs

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.prototypes = []
        self.prototype_labels = []

        print("\nInitial Prototype Weights:")
        for cls in self.classes:
            class_data = X[y == cls]
            for i in range(self.n_prototypes_per_class):
                prototype = class_data[np.random.randint(0, len(class_data))]
                self.prototypes.append(prototype)
                self.prototype_labels.append(cls)
                print(f"Prototype for class {cls} ({class_names[cls]}): {prototype}")

        self.prototypes = np.array(self.prototypes)

        for epoch in range(self.epochs):
            for i, x in enumerate(X):
                distances = np.linalg.norm(self.prototypes - x, axis=1)
                winner_idx = np.argmin(distances)

                if self.prototype_labels[winner_idx] == y[i]:
                    self.prototypes[winner_idx] += self.learning_rate * (x - self.prototypes[winner_idx])
                else:
                    self.prototypes[winner_idx] -= self.learning_rate * (x - self.prototypes[winner_idx])

        print("\nFinal Prototype Weights:")
        for i, proto in enumerate(self.prototypes):
            print(f"Prototype {i} → Class {self.prototype_labels[i]} ({class_names[self.prototype_labels[i]]})")
            print("Weights:", proto)

    def predict(self, X):
        predictions = []
        for x in X:
            distances = np.linalg.norm(self.prototypes - x, axis=1)
            winner_idx = np.argmin(distances)
            predictions.append(self.prototype_labels[winner_idx])
        return np.array(predictions)



In [5]:
lvq = LVQ(n_prototypes_per_class=2, learning_rate=0.05, epochs=100)

print("\nTraining Vectors Sample:")
print(X_train[:5])

lvq.fit(X_train, y_train)

y_pred = lvq.predict(X_test)

accuracy = np.mean(y_pred == y_test)
print("\nModel Accuracy:", accuracy)

print("\nSample Predictions:")
for i in range(5):
    print(f"Actual: {class_names[y_test[i]]}, Predicted: {class_names[y_pred[i]]}")


Training Vectors Sample:
[[0.19973389 0.8106969  0.19469858]
 [0.14557461 0.79396679 0.14598279]
 [0.18928642 0.80733951 0.17972176]
 [0.32579342 0.79864561 0.32487197]
 [0.30632762 0.79872483 0.30088196]]

Initial Prototype Weights:
Prototype for class 0 (DOWN (Sell Signal)): [0.36738616 0.80589392 0.35673279]
Prototype for class 0 (DOWN (Sell Signal)): [0.03927656 0.77677406 0.03656589]
Prototype for class 1 (UP (Buy Signal)): [0.27833629 0.81190467 0.26176024]
Prototype for class 1 (UP (Buy Signal)): [0.28065247 0.81897466 0.26431607]

Final Prototype Weights:
Prototype 0 → Class 0 (DOWN (Sell Signal))
Weights: [0.50956645 0.70844943 0.53849602]
Prototype 1 → Class 0 (DOWN (Sell Signal))
Weights: [0.14747566 0.74944687 0.15353032]
Prototype 2 → Class 1 (UP (Buy Signal))
Weights: [0.09565771 0.93593678 0.07698338]
Prototype 3 → Class 1 (UP (Buy Signal))
Weights: [0.3187338  0.91161353 0.27858522]

Model Accuracy: 0.6036585365853658

Sample Predictions:
Actual: DOWN (Sell Signal), Pr